# Find samples using YARA Hunting and send them to Flexible Intel Feed

This notebook showcases how to search for relevant samples using YARA hunting and send them to Flexible Intel Feed.
A flow like this one gives us a simple yet useful way to utilize our custom criteria for selectively sending samples to Flexible Intel Feed analysis.
To make this workflow send samples to Flexible Intel Feed, we need to have Flexible Intel Feed enabled on our Spectra Analyze instance.

### Used Spectra Analyze functions
- **create_or_update_yara_ruleset**
- **start_or_stop_yara_cloud_retro_scan**
- **get_yara_cloud_retro_scan_status**
- **get_yara_ruleset_matches_v2**
- **reanalyze_samples_v2**

### Credentials
Credentials are loaded from a local file instead of being written here in plain text.
To learn how to create credentials file, see the **Storing and using credentials** section in the [README file](./README.md)


### 1. Hunt for samples using YARA
First, we have to do the required Python imports and load our credentials. After that, we create our A1000 object.

In [ ]:
import json
from ReversingLabs.SDK.a1000 import A1000

CREDENTIALS = json.load(open('credentials.json'))
HOST = CREDENTIALS.get("a1000").get("a1000_url")
TOKEN = CREDENTIALS.get("a1000").get("token")
USER_AGENT = json.load(open('../user_agent.json'))["user_agent"]

a1000_obj = A1000(
    host=HOST,
    token=TOKEN,
    user_agent=USER_AGENT,
    verify=False
)

Now we can perform YARA hunting to finds samples we want to send to Flexible Intel Feed.
Change the `RULESET_NAME` and the `RULESET_CONTENT` to fit your sample hunting needs. The given ruleset example hunts for `Trojan.AndroidOS.SMSThief` malware in the ReversingLabs cloud.

In [ ]:
RULESET_NAME = "AndroidTrojanSMSThief"
RULESET_CONTENT = f"""
rule {RULESET_NAME}
{{
    meta:
        description = "Detects Android SMSThief trojans based on permission requests and receiver actions"
        family = "Trojan.AndroidOS.SMSThief"
        severity = "Critical"

    strings:
        $perm_sms_recv = "android.permission.RECEIVE_SMS"
        $perm_sms_read = "android.permission.READ_SMS"
        $perm_sms_send = "android.permission.SEND_SMS"
        $perm_net = "android.permission.INTERNET"

        $action_sms_received = "android.provider.Telephony.SMS_RECEIVED"
        $action_boot = "android.intent.action.BOOT_COMPLETED"

        $str_intercept = "sms_intercept" nocase
        $str_forward = "forward_sms" nocase

    condition:
        uint32(0) == 0x04034b50 and

        ($perm_sms_recv and $perm_net) and

        ($action_sms_received or $action_boot) and

        (2 of ($perm_sms_read, $perm_sms_send, $str_intercept, $str_forward))
}}
"""

resp = a1000_obj.create_or_update_yara_ruleset(
    name=RULESET_NAME,
    content=RULESET_CONTENT,
    ticloud=True
)

print(resp.text)


Next, we have to start YARA Retro scan using our newly created ruleset.

In [ ]:
resp = a1000_obj.start_or_stop_yara_cloud_retro_scan(
    operation="START",
    ruleset_name=RULESET_NAME
)

print(resp.text)

We need to periodically check the status of our ongoing YARA Retro scan to see if the scan is still running or if it finished.

In [ ]:
resp = a1000_obj.get_yara_cloud_retro_scan_status(
    ruleset_name=RULESET_NAME
)

print(resp.text)

Even if our scan is still running there may already be a lot of results available.
We can try and fetch them using the following method.

In [ ]:
matches_resp = a1000_obj.get_yara_ruleset_matches_v2(
    ruleset_name=RULESET_NAME
)

print(resp.text)

If successful, this action gave us a list of Python dicts. Now we can iterate through that list and request sample reanalysis for each entry.

### 2. Reanalyze samples to send them to Flexible Intel Feed
Make sure to set `titanium_cloud` to `True` as uploading the sample to the cloud is required for triggering Flexible Intel Feed analysis.

In [ ]:
sha1_list = []

for sample in matches_resp.json().get("results"):
    sha1_list.append(sample.get("sha1"))

reanalyze_resp = a1000_obj.reanalyze_samples_v2(
    hash_input=sha1_list,
    titanium_cloud=True
)
print(reanalyze_resp.text)

### 2. Stopping the YARA hunt
If we want to stop our YARA hunt scan, we can do so by running the following options:



In [ ]:
resp = a1000_obj.start_or_stop_yara_cloud_retro_scan(
    operation="STOP",
    ruleset_name=RULESET_NAME
)

print(resp.text)